Parse data into propaganda fragments with corresponding propaganda techniques

In [1]:
import pandas as pd
import ast

In [2]:
df = pd.read_csv("data\\concatenated.csv")

In [3]:
df.head()

,Unnamed: 0,annotation_id,annotator,content,created_at,customDesinformationTechnique,desinformationTechnique,heading,id,label,lead_time,primaryChoice,updated_at
0,0,828,14,Visų pirma nusiimkim spalvotus vaikiškus akinė...,2024-01-11T10:50:03.931836Z,NaN,"{""choices"":[""distrustOfLithuanianInstitutions""...",Algimantas Rusteika. Paktas,37735,"[{'start': 0, 'end': 360, 'labels': ['simplifi...",13048.712,yes,2024-01-11T10:50:03.931849Z
1,1,829,14,Nors Lietuvos profesinė sąjunga „Sandrauga“ da...,2024-01-11T10:50:03.931979Z,NaN,lithuanianDefamation,Dveji prezidentavimo metai dirbančiųjų akimis,37736,"[{'start': 131, 'end': 187, 'labels': ['emotio...",8894.908,yes,2024-01-11T10:50:03.931987Z
2,2,830,14,Nepriklausomas gerų žinių portalas minfo.lt pa...,2024-01-11T10:50:03.932120Z,NaN,lithuanianDefamation,Peticija: Reikalaujame nušalinti 15min nuo „Fa...,37737,"[{'start': 76, 'end': 148, 'labels': ['emotion...",12202.753,yes,2024-01-11T10:50:03.932128Z
3,3,831,14,\nŽmonės šiai valdžiai nerūpi. Jiems nesvarbi ...,2024-01-11T10:50:03.932281Z,NaN,distrustOfLithuanianInstitutions,Ramūnas Karbauskis: Konservatoriai su liberala...,37738,"[{'start': 958, 'end': 1515, 'labels': ['doubt...",1714.853,yes,2024-01-11T10:50:03.932289Z
4,4,923,11,"Tegul jie tą seimą ir gina. Kiek jų yra, parod...",2024-01-12T13:42:00.575907Z,NaN,distrustOfLithuanianInstitutions,"Viktoras Jašinskas: LR Seimas parodė, kad atst...",37744,"[{'start': 180, 'end': 378, 'labels': ['emotio...",599.352,yes,2024-01-12T13:42:00.575917Z


In [4]:
df = df.loc[:, ["annotation_id", "content", "label"]]
print(df.info())
# Remove na
df.dropna(inplace=True)
print("unique ids:", df["annotation_id"].nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1822 entries, 0 to 1821
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   annotation_id  1822 non-null   int64 
 1   content        1822 non-null   object
 2   label          911 non-null    object
dtypes: int64(1), object(2)
memory usage: 42.8+ KB
None
unique ids: 911


In [5]:
with pd.option_context('display.max_colwidth', None):  # more options can be specified also
    print(df["label"].head())

0    [{'start': 0, 'end': 360, 'labels': ['simplification']}, {'start': 908, 'end': 1088, 'labels': ['simplification']}, {'start': 1089, 'end': 1454, 'labels': ['emotionalExpression']}, {'start': 1654, 'end': 1950, 'labels': ['uncertainty']}, {'start': 1951, 'end': 2416, 'labels': ['uncertainty']}, {'start': 2851, 'end': 2901, 'labels': ['emotionalExpression']}, {'start': 2446, 'end': 2650, 'labels': ['uncertainty']}, {'start': 2903, 'end': 3387, 'labels': ['emotionalExpression']}, {'start': 3388, 'end': 3418, 'labels': ['emotionalExpression']}, {'start': 3388, 'end': 3416, 'labels': ['simplification']}, {'start': 4597, 'end': 4926, 'labels': ['emotionalExpression']}, {'start': 5440, 'end': 5968, 'labels': ['emotionalExpression']}, {'start': 5458, 'end': 5653, 'labels': ['doubt']}, {'start': 5969, 'end': 6091, 'labels': ['doubt']}, {'start': 6051, 'end': 6089, 'labels': ['wavingTheFlag']}, {'start': 6091, 'end': 6589, 'labels': ['emotionalExpression']}, {'start': 6611, 'end': 6791, 'la

In [6]:
fragments = []
for index, row in df.iterrows():
    content: str = row["content"] # format: string
    id: int = row["annotation_id"]
    labels_list: list = ast.literal_eval(row["label"]) # format: list
    for dictionary in labels_list:
        start = dictionary.get("start")
        end = dictionary.get("end")
        labels = dictionary.get("labels") 
        labels = "".join(labels)
        fragments.append((id, content[start:end+1], labels)) 

new_df = pd.DataFrame(data=fragments, columns=["id", "fragment", "label"])

In [8]:
# Print the unique propaganda techniques
techniques = pd.unique(new_df["label"])
print("propaganda techniques", techniques)
print("number of techniques", len(techniques))
# empty technique?
print(len(new_df[new_df['label'] == ""]))
print(new_df[new_df['label'] == ""])
new_df = new_df[new_df['label'] != ""]

print(new_df.shape)
new_df = new_df[new_df["fragment"].str.strip() != ""] # removing empty fragment strings
print(new_df.shape)
new_df.to_csv("data\\prepared_data.csv", index=False)

propaganda techniques ['simplification' 'emotionalExpression' 'uncertainty' 'doubt'
 'wavingTheFlag' 'reductioAdHitlerum' 'repetition' 'appealToAuthority'
 'whataboutismRedHerringStrawMan' 'followingBehind']
number of techniques 10
0
Empty DataFrame
Columns: [id, fragment, label]
Index: []
(13652, 3)
(13652, 3)


In [14]:
# Test with small subset
sub_new_df = new_df.iloc[:100, :]
sub_new_df.shape
sub_new_df.to_csv("data\\sub_prepared_data.csv", index=False)